In [1]:
import pandas as pd

In [2]:
df = pd.DataFrame({
    "user_id": ["u1", "u1", "u1", "u1", "u2", "u2", "u2", "u3"],
    "event_type": ["create", "edit", "edit", "export", "create", "create", "share", "export"],
    "timestamp": pd.to_datetime([
        "2024-01-01 10:00", "2024-01-01 11:00", "2024-01-02 09:00", "2024-01-02 14:00",
        "2024-01-01 08:00", "2024-01-01 09:00", "2024-01-03 10:00", "2024-01-05 12:00",
    ]),
    "design_id": ["d1", "d1", "d2", "d2", "d3", "d3", "d4", "d5"],
})

In [3]:
df

,user_id,event_type,timestamp,design_id
0,u1,create,2024-01-01 10:00:00,d1
1,u1,edit,2024-01-01 11:00:00,d1
2,u1,edit,2024-01-02 09:00:00,d2
3,u1,export,2024-01-02 14:00:00,d2
4,u2,create,2024-01-01 08:00:00,d3
5,u2,create,2024-01-01 09:00:00,d3
6,u2,share,2024-01-03 10:00:00,d4
7,u3,export,2024-01-05 12:00:00,d5


In [9]:
df.iloc[0].timestamp.date()

datetime.date(2024, 1, 1)

In [11]:
df['date'] = df.apply(lambda r: r.timestamp.date(), axis=1)
df

,user_id,event_type,timestamp,design_id,date
0,u1,create,2024-01-01 10:00:00,d1,2024-01-01
1,u1,edit,2024-01-01 11:00:00,d1,2024-01-01
2,u1,edit,2024-01-02 09:00:00,d2,2024-01-02
3,u1,export,2024-01-02 14:00:00,d2,2024-01-02
4,u2,create,2024-01-01 08:00:00,d3,2024-01-01
5,u2,create,2024-01-01 09:00:00,d3,2024-01-01
6,u2,share,2024-01-03 10:00:00,d4,2024-01-03
7,u3,export,2024-01-05 12:00:00,d5,2024-01-05


In [ ]:
# - total_events: int - total number of events per user
#    - unique_designs: int - number of distinct designs per user
#    - most_common_event: str - the event_type that appears most often for each user
#                               (if tied, any of the tied values is acceptable)
#    - days_active: int - number of distinct calendar dates the user had events on
#    - export_rate: float - fraction of the user's events that are "export" events
#                           (0.0 if user has no exports)

In [48]:
from collections import Counter

def _compute_features(r: pd.Series) -> tuple[int, str, int, float]:
    num_unique_designs = len(r.design_id)
    cl = Counter(r.event_type)
    most_common_event = cl.most_common()[0][0]
    days_active = len(r.date)
    num_events = len(r.event_type)
    export_rate = 0.0
    if num_events > 0:
        export_rate = cl['export'] / len(r.event_type)
    return num_unique_designs, most_common_event, days_active, export_rate

user_aggs = df[
    ['user_id', 'event_type', 'design_id', 'date']
].groupby(['user_id']).agg(
    {
        'design_id': set,
        'date': set,
        'event_type': list,
    }
)#.reset_index()
user_aggs[[
    "unique_designs", "most_common_event", "days_active", "export_rate",
]] = user_aggs.apply(
    lambda r: _compute_features(r),
    axis=1,
    result_type='expand',
)
out = user_aggs[["unique_designs", "most_common_event", "days_active", "export_rate"]]
out

,unique_designs,most_common_event,days_active,export_rate
user_id,,,,
u1,2,edit,2,0.25
u2,2,create,2,0.00
u3,1,export,1,1.00


In [42]:
user_aggs[[
    "unique_designs", "most_common_event", "days_active", "export_rate",
]] = user_aggs.apply(
    lambda r: _compute_features(r),
    axis=1,
    result_type='expand',
)
out = user_aggs[["unique_designs", "most_common_event", "days_active", "export_rate",]]

In [18]:
l = ['b', 'a', 'c', 'c', 'd']
l

['b', 'a', 'c', 'c', 'd']

In [19]:
from collections import Counter

In [26]:
cl = Counter(l)
cl

Counter({'c': 2, 'b': 1, 'a': 1, 'd': 1})

In [36]:
cl

Counter({'c': 2, 'b': 1, 'a': 1, 'd': 1})

In [44]:
cl['c']

2

In [28]:
mc = cl.most_common()[0]
mc

('c', 2)

In [ ]:
user_aggs[
    
]

In [50]:
df

,user_id,event_type,timestamp,design_id,date
0,u1,create,2024-01-01 10:00:00,d1,2024-01-01
1,u1,edit,2024-01-01 11:00:00,d1,2024-01-01
2,u1,edit,2024-01-02 09:00:00,d2,2024-01-02
3,u1,export,2024-01-02 14:00:00,d2,2024-01-02
4,u2,create,2024-01-01 08:00:00,d3,2024-01-01
5,u2,create,2024-01-01 09:00:00,d3,2024-01-01
6,u2,share,2024-01-03 10:00:00,d4,2024-01-03
7,u3,export,2024-01-05 12:00:00,d5,2024-01-05


In [51]:
grouped = df.groupby('user_id')
grouped

In [52]:
total_events = grouped['event_type'].count()
total_events

user_id
u1    4
u2    3
u3    1
Name: event_type, dtype: int64

In [54]:
unique_designs = grouped['design_id'].nunique()
unique_designs

user_id
u1    2
u2    2
u3    1
Name: design_id, dtype: int64

In [57]:
most_common_event = grouped['event_type'].agg(lambda x: x.mode().iloc[0]) # the extra iloc[0] if there are multiple ties
most_common_event

user_id
u1      edit
u2    create
u3    export
Name: event_type, dtype: object

In [58]:
days_active = grouped['timestamp'].agg(lambda x: x.dt.date.nunique())
days_active

user_id
u1    2
u2    2
u3    1
Name: timestamp, dtype: int64

In [59]:
export_counts = df[df['event_type'] == 'export'].groupby('user_id').size()
export_counts

user_id
u1    1
u3    1
dtype: int64

In [60]:
export_rate = (export_counts / total_events).fillna(0.0)
export_rate

user_id
u1    0.25
u2    0.00
u3    1.00
dtype: float64

In [61]:
result = pd.DataFrame({
    'total_events': total_events,
    'unique_designs': unique_designs,
    'most_common_event': most_common_event,
    'days_active': days_active,
    'export_rate': export_rate,
})
result.index.name = 'user_id'

In [62]:
result

,total_events,unique_designs,most_common_event,days_active,export_rate
user_id,,,,,
u1,4,2,edit,2,0.25
u2,3,2,create,2,0.00
u3,1,1,export,1,1.00
